In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# data read
train = pd.read_csv("/kaggle/input/competitions/triagegeist/train.csv")
chief = pd.read_csv("/kaggle/input/competitions/triagegeist/chief_complaints.csv")
hist = pd.read_csv("/kaggle/input/competitions/triagegeist/patient_history.csv")
test = pd.read_csv("/kaggle/input/competitions/triagegeist/test.csv")
df = train.merge(chief, on="patient_id", how="left")
df = df.merge(hist, on="patient_id", how="left")
df_test=test.merge(chief, on="patient_id", how="left")
df_test=df_test.merge(hist, on="patient_id", how="left")

df.info()
df_test.info()

y = df["triage_acuity"]
X = df.drop(["triage_acuity","patient_id"], axis=1)


# analysys of 'chief_complaint_raw'

# Is there the same text?
print(df['chief_complaint_raw'].nunique())
# No of the same text
print(df['chief_complaint_raw'].value_counts())
# same text has the same triage level?
print(df.groupby('chief_complaint_raw')['triage_acuity'].nunique())
# How many triage level do the same text have?
print((df.groupby('chief_complaint_raw')['triage_acuity'].nunique()==2).sum())
print((df.groupby('chief_complaint_raw')['triage_acuity'].nunique()==1).sum())



# Is there the same text?
print(df_test['chief_complaint_raw'].nunique())
# No of the same text
print(df_test['chief_complaint_raw'].value_counts())







* missing datas are ralatively a few.
*text of chif_complaint_raw is not unique,and almost text have same value.
variation of text is 4949 in 10000 texts.
this tendency is observed test data.



In [ ]:
#LBGM using categorized text data and other features by Group Kfold cross varidation analysis 
import pandas as pd
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Restore X and y to their original state to ensure consistency
y = df["triage_acuity"]
X = df.drop(["triage_acuity","patient_id"], axis=1)
print("X and y dataframes restored to their original state.")


# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# List to store accuracy scores for each fold
accuracy_scores = [] # Initialize accuracy_scores list
all_feature_importances = []

print("\nStarting GroupKFold cross-validation for feature importance calculation...")

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)):
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, y_train_fold = X.iloc[train_index].copy(), y.iloc[train_index].copy()
    X_valid_fold, y_valid_fold = X.iloc[val_index].copy(), y.iloc[val_index].copy()

    # Corrected: Use X_train_fold and X_valid_fold directly
    for col in X_train_fold.select_dtypes(include="object").columns:
      X_train_fold[col] = X_train_fold[col].astype("category")

    for col in X_valid_fold.select_dtypes(include="object").columns:
      X_valid_fold[col] = X_valid_fold[col].astype("category")


    # Initialize and train the LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5, # Assuming 5 classes for triage_acuity
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    model.fit(X_train_fold, y_train_fold)

    pred = model.predict(X_valid_fold)

    # j. Calculate accuracy and append
    accuracy = accuracy_score(y_valid_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

    # Get feature importances for the current fold and append to the list
    all_feature_importances.append(model.feature_importances_)
    print(f"Fold {fold+1} model trained and feature importances collected.")

# Calculate mean and standard deviation
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelBinarizer
import numpy as np

# --- Confusion Matrix ---
print("\n--- Confusion Matrix (Last Fold) ---")
cm = confusion_matrix(y_valid_fold, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
# Calculate the average feature importances across all folds
mean_feature_importances = np.mean(all_feature_importances, axis=0)

# Create a DataFrame for average feature importances
feature_names = X.columns.tolist()
importance_df_gkf = pd.DataFrame({
    'Feature': feature_names,
    'Importance': mean_feature_importances
}).sort_values(by='Importance', ascending=False)

# Print top 20 features
N = 20
print(f"\nTop {N} Average Feature Importances (GroupKFold):")
print(importance_df_gkf.head(N))

# Visualize top 20 features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_gkf.head(N), palette='viridis')
plt.title(f'Top {N} Average Feature Importances (GroupKFold LGBMClassifier)')
plt.xlabel('Average Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

Categoralized data and LGBM showed average accuracy rate 0.8575

In [ ]:
#LBGM using TF-IDF text data and other features by Group Kfold cross varidation analysisGroup
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Restore X and y to their original state (assuming 'df' is available globally)
y = df["triage_acuity"]
X = df.drop(["triage_acuity", "patient_id"], axis=1)

print("X and y dataframes restored to their original state.")

# Define the groups for GroupKFold using the original 'chief_complaint_raw' column
# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# 1. Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# 2. Create empty lists to store accuracy scores and feature importances
accuracy_scores = []
all_feature_importances = []
feature_names_list = [] # To store feature names for each fold

# 3. Initialize and fit TfidfVectorizer on the entire 'chief_complaint_raw' column once
#    This ensures a consistent feature space for TF-IDF across all folds
tv = TfidfVectorizer(stop_words='english')


print("Starting GroupKFold cross-validation...")

# 4. Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)): # Use gkf.split
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index].copy(), y.iloc[val_index].copy()

    # b. Apply TF-IDF transformation (using globally fitted tv)

    X_train_tfidf_sparse = tv.fit_transform(X_train_fold['chief_complaint_raw'])
    X_val_tfidf_sparse = tv.transform(X_val_fold['chief_complaint_raw'])

    # Get TF-IDF feature names after fitting on the training data
    tfidf_feature_names = tv.get_feature_names_out()

    # c. Convert sparse TF-IDF matrices to pandas DataFrames
    X_train_tfidf_df = pd.DataFrame(X_train_tfidf_sparse.toarray(),
                                    index=X_train_fold.index,
                                    columns=tfidf_feature_names)
    X_val_tfidf_df = pd.DataFrame(X_val_tfidf_sparse.toarray(),
                                  index=X_val_fold.index,
                                  columns=tfidf_feature_names)

    # d. Drop the original 'chief_complaint_raw' column from the other features
    X_train_other_features = X_train_fold.drop('chief_complaint_raw', axis=1)
    X_val_other_features = X_val_fold.drop('chief_complaint_raw', axis=1)

    # e. Convert remaining object type columns to 'category' dtype
    cat_cols_train = X_train_other_features.select_dtypes(include="object").columns
    for c in cat_cols_train:
        X_train_other_features[c] = X_train_other_features[c].astype("category")

    cat_cols_val = X_val_other_features.select_dtypes(include="object").columns
    for c in cat_cols_val:
        X_val_other_features[c] = X_val_other_features[c].astype("category")

    # f. Concatenate all features (other features + TF-IDF features)
    X_train_final = pd.concat([X_train_other_features, X_train_tfidf_df], axis=1)
    X_val_final = pd.concat([X_val_other_features, X_val_tfidf_df], axis=1)

    # g. Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5,
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # h. Train the model
    model.fit(X_train_final, y_train_fold)

    # i. Make predictions
    pred = model.predict(X_val_final)

    # j. Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

    # k. Store feature importances and names for this fold
    all_feature_importances.append(model.feature_importances_)
    feature_names_list.append(X_train_final.columns.tolist())

# 5. Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Print the results
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

# --- Confusion Matrix ---
print("\n--- Confusion Matrix (Last Fold) ---")
cm = confusion_matrix(y_val_fold, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# 6. Calculate average feature importances
# Assuming feature names are consistent across folds, use the names from the last fold
# Or, if feature sets can differ, align them before averaging.
# For this case, we'll assume they are consistent enough or that taking the last fold's names is representative.
if len(all_feature_importances) > 0 and len(feature_names_list) > 0:
    # Create a DataFrame for feature importances from each fold
    feature_importance_dfs = []
    for i, importances in enumerate(all_feature_importances):
        df_imp = pd.DataFrame({'Feature': feature_names_list[i], 'Importance': importances})
        feature_importance_dfs.append(df_imp)

    # Concatenate all importance DataFrames and calculate the mean importance for each feature
    combined_importance_df = pd.concat(feature_importance_dfs)
    importance_df_gkf = combined_importance_df.groupby('Feature')['Importance'].mean().reset_index()
    importance_df_gkf = importance_df_gkf.sort_values(by='Importance', ascending=False)
else:
    print("No feature importances to display. Cross-validation might not have run.")
    importance_df_gkf = pd.DataFrame(columns=['Feature', 'Importance']) # Create empty DF to prevent further errors


# Print top 20 features
N = 20
print(f"\nTop {N} Average Feature Importances (GroupKFold):")
print(importance_df_gkf.head(N))

# 10. Visualize top 20 features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_gkf.head(N), palette='viridis')
plt.title(f'Top {N} Average Feature Importances (GroupKFold LGBMClassifier)')
plt.xlabel('Average Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

LBGM analysis TF-IDF from 'chief_complaint_raw'and other data showed average accuracy 0.9991.

## Summary of Feature Importances

### Q&A
The most impactful features from the analysis include both numerical/categorical features from the original dataset and terms extracted via TF-IDF from the 'chief_complaint_raw' text.

### Data Analysis Key Findings
Based on the LGBMClassifier's feature importance (gain), the top 20 features are:

**Numerical/Categorical Features from Original Dataset:**
*   `ed_los_hours`
*   `respiratory_rate`
*   `spo2`
*   `pain_score`
*   `temperature_c`
*   `mean_arterial_pressure`
*   `heart_rate`
*   `diastolic_bp`
*   `news2_score`
*   `num_prior_ed_visits_12m`
*   `height_cm`
*   `pulse_pressure`
*   `shock_index`
*   `systolic_bp`
*   `gcs_total` 
*   `age` 

**TF-IDF Processed Text Features:**
*   `mild`: 
*   `acute`: 
*   `severe`: 
*   `moderate`: 

This distribution indicates that several physiological measurements (`ed_los_hours`, `respiratory_rate`, `spo2`, `pain_score`, `temperature_c`, `mean_arterial_pressure`, `heart_rate`, `diastolic_bp`), patient history (`num_prior_ed_visits_12m`, `age`), and overall health scores (`news2_score`, `gcs_total`, `shock_index`) are highly influential. Crucially, the TF-IDF terms like 'mild', 'acute', 'severe', and 'moderate' also play a significant role, suggesting that the descriptive intensity of the chief complaint is very important for predicting `triage_acuity`.




In [ ]:
#LBGM with TF-IDF text data only 
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Restore X and y to their original state (assuming 'df' is available globally)
y = df["triage_acuity"]
X = df.drop(["triage_acuity", "patient_id"], axis=1)

print("X and y dataframes restored to their original state.")

# Define the groups for GroupKFold using the 'chief_complaint_raw' column
# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# List to store accuracy scores for each fold
accuracy_scores = []

# Initialize TfidfVectorizer
tv = TfidfVectorizer(stop_words='english')

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)):
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index].copy(), y.iloc[val_index].copy()

    # Apply TF-IDF transformation to 'chief_complaint_raw' only
    # Fit TfidfVectorizer on training data only to avoid data leakage
    X_train_tfidf_sparse = tv.fit_transform(X_train_fold['chief_complaint_raw'])
    X_val_tfidf_sparse = tv.transform(X_val_fold['chief_complaint_raw'])

    # Convert sparse TF-IDF matrices to pandas DataFrames
    X_train_final = pd.DataFrame(X_train_tfidf_sparse.toarray(),
                                    index=X_train_fold.index,
                                    columns=tv.get_feature_names_out())
    X_val_final = pd.DataFrame(X_val_tfidf_sparse.toarray(),
                                  index=X_val_fold.index,
                                  columns=tv.get_feature_names_out())

    # Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5, # Assuming 5 classes for triage_acuity
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # Train the model using only TF-IDF features
    model.fit(X_train_final, y_train_fold)

    # Make predictions
    pred = model.predict(X_val_final)

    # Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

# Calculate mean and standard deviation
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Print the results
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

LBGM analysis from TH-IDF data from 'chief_complaint_raw' only showed average accuracy 0.9975.

## Summary of TF-IDF Feature Importances

### Q&A
The most impactful features for predicting `triage_acuity` when using only TF-IDF features are specific terms extracted from the 'chief_complaint_raw' text, such as 'acute', 'mild', 'fever', 'worsening', and 'severe'.

### Data Analysis Key Findings
Based on the LGBMClassifier's feature importance (gain) from only TF-IDF features, the top 20 most impactful terms are:

*   `acute`: 5930
*   `mild`: 5753
*   `fever`: 4612
*   `worsening`: 4414
*   `severe`: 4298
*   `history`: 3268
*   `vomiting`: 3168
*   `diaphoresis`: 2925
*   `moderate`: 2398
*   `yesterday`: 2247
*   `minor`: 2161
*   `constant`: 2115
*   `days`: 2036
*   `associated`: 1980
*   `rigors`: 1901
*   `onset`: 1891
*   `intermittent`: 1844
*   `known`: 1834
*   `nausea`: 1696
*   `patient`: 1652

This highlights that descriptive terms related to the severity, nature, and temporal aspects of the chief complaint are highly influential in predicting `triage_acuity`.



In [ ]:
#Perform SHAP Analysis for TF-IDF Features
import shap
import matplotlib.pyplot as plt

# 1. Initialize a SHAP TreeExplainer with the trained model
explainer = shap.TreeExplainer(model)

# 2. Create a sample of X_val_final to manage computation time for SHAP values
X_val_sample = X_val_final.sample(n=1000, random_state=42)

# 3. Calculate the SHAP values for X_val_sample
# For multiclass classification, shap_values will be a list of arrays, one for each class.
shap_values = explainer.shap_values(X_val_sample)

# 4. Generate a SHAP summary plot of type 'bar'
# When shap_values is a list (multiclass), shap.summary_plot automatically handles it
# to show overall feature importance across classes.
shap.summary_plot(shap_values, X_val_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance for TF-IDF Features")
plt.tight_layout()
plt.show()



In [ ]:
#LGBM with TF-IDF text data and 6 other vital features
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Restore X and y to their original state (assuming 'df' is available globally)
y = df["triage_acuity"]
X = df.drop(["triage_acuity", "patient_id"], axis=1)

print("X and y dataframes restored to their original state.")

# Define the groups for GroupKFold using the 'chief_complaint_raw' column
# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# List to store accuracy scores for each fold
accuracy_scores = []

# Initialize TfidfVectorizer
tv = TfidfVectorizer(stop_words='english')

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)):
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index].copy(), y.iloc[val_index].copy()

    # Apply TF-IDF transformation to 'chief_complaint_raw' only
    # Fit TfidfVectorizer on training data only to avoid data leakage
    X_train_tfidf_sparse = tv.fit_transform(X_train_fold['chief_complaint_raw'])
    X_val_tfidf_sparse = tv.transform(X_val_fold['chief_complaint_raw'])

    # Convert sparse TF-IDF matrices to pandas DataFrames
    X_train_tfidf = pd.DataFrame(X_train_tfidf_sparse.toarray(),
                                    index=X_train_fold.index,
                                    columns=tv.get_feature_names_out())
    X_val_tfidf = pd.DataFrame(X_val_tfidf_sparse.toarray(),
                                  index=X_val_fold.index,
                                  columns=tv.get_feature_names_out())
    # Corrected pd.concat and column selection syntax
    X_train_final = pd.concat([X_train_tfidf, X_train_fold[['respiratory_rate','spo2','temperature_c','heart_rate','diastolic_bp','mean_arterial_pressure']]], axis=1)
    X_val_final = pd.concat([X_val_tfidf, X_val_fold[['respiratory_rate','spo2','temperature_c','heart_rate','diastolic_bp','mean_arterial_pressure']]], axis=1)

    # Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5, # Assuming 5 classes for triage_acuity
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # Train the model using only TF-IDF features
    model.fit(X_train_final, y_train_fold)

    # Make predictions
    pred = model.predict(X_val_final)

    # Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

# Calculate mean and standard deviation
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Print the results
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelBinarizer
import numpy as np

# --- Confusion Matrix ---
print("\n--- Confusion Matrix (Last Fold) ---")
cm = confusion_matrix(y_val_fold, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

df_test_tfidf_sparse = tv.transform(df_test['chief_complaint_raw'])

    

df_test_tfidf = pd.DataFrame(df_test_tfidf_sparse.toarray(),
                                  index=df_test.index,
                                  columns=tv.get_feature_names_out())
    
df_test_final = pd.concat([df_test_tfidf, df_test[['respiratory_rate','spo2','temperature_c','heart_rate','diastolic_bp','mean_arterial_pressure']]], axis=1)





df_test['triage_acuity'] = model.predict(df_test_final)

submission = df_test[['patient_id', 'triage_acuity']]


submission.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the pre-trained 'all-MiniLM-L6-v2' model
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the 'chief_complaint_raw' column
X['chief_complaint_raw_embedding'] = X['chief_complaint_raw'].apply(lambda x: model_sbert.encode(x))

print("Sentence embeddings generated for 'chief_complaint_raw' column.")
print(f"Shape of the first embedding: {X['chief_complaint_raw_embedding'].iloc[0].shape}")



In [ ]:
#LGBM with sentence embedded text data and other featuers using Group Kfold cross variation method
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
# from sklearn.feature_extraction.text import TfidfVectorizer # Not needed for embeddings directly here
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer # Needed to generate embeddings if X is restored without them

# Restore X and y to their original state (assuming 'df' is available globally)
y = df["triage_acuity"]
X = df.drop(["triage_acuity", "patient_id"], axis=1)

print("X and y dataframes restored to their original state.")

# Ensure 'chief_complaint_raw_embedding' column exists in X
if 'chief_complaint_raw_embedding' not in X.columns:
    print("Generating sentence embeddings for 'chief_complaint_raw'...")
    model_sbert = SentenceTransformer('all-MiniLM-L6-v2')
    X['chief_complaint_raw_embedding'] = X['chief_complaint_raw'].apply(lambda x: model_sbert.encode(x))
    print("Sentence embeddings generated.")

# Define the groups for GroupKFold using the original 'chief_complaint_raw' column
# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# Create empty lists to store accuracy scores and feature importances
accuracy_scores = []
all_feature_importances = []

# Get the number of dimensions for the embeddings
embedding_dimensions = X['chief_complaint_raw_embedding'].iloc[0].shape[0]
embedding_feature_names = [f'embedding_{i}' for i in range(embedding_dimensions)]

# Get feature names for non-embedding features (excluding 'chief_complaint_raw' as it's processed into embeddings)
other_feature_names = X.drop(['chief_complaint_raw', 'chief_complaint_raw_embedding'], axis=1).columns.tolist()

# Combine all feature names for later use in importance visualization
full_feature_names = other_feature_names + embedding_feature_names

print("Starting GroupKFold cross-validation with sentence embeddings...")

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)): # Use gkf.split with full X and y
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index].copy(), y.iloc[val_index].copy()

    # Create DataFrames from the sentence embedding Series for train and validation
    X_train_embedding_df = pd.DataFrame(
        X_train_fold['chief_complaint_raw_embedding'].tolist(),
        index=X_train_fold.index,
        columns=embedding_feature_names
    )
    X_val_embedding_df = pd.DataFrame(
        X_val_fold['chief_complaint_raw_embedding'].tolist(),
        index=X_val_fold.index,
        columns=embedding_feature_names
    )

    # Drop the original 'chief_complaint_raw' and 'chief_complaint_raw_embedding' columns
    X_train_other_features = X_train_fold.drop(['chief_complaint_raw', 'chief_complaint_raw_embedding'], axis=1)
    X_val_other_features = X_val_fold.drop(['chief_complaint_raw', 'chief_complaint_raw_embedding'], axis=1)

    # Convert remaining object type columns to 'category' dtype
    cat_cols_train = X_train_other_features.select_dtypes(include="object").columns
    for c in cat_cols_train:
        X_train_other_features[c] = X_train_other_features[c].astype("category")

    cat_cols_val = X_val_other_features.select_dtypes(include="object").columns
    for c in cat_cols_val:
        X_val_other_features[c] = X_val_other_features[c].astype("category")

    # Concatenate all features (other features + embedding features)
    X_train_final = pd.concat([X_train_other_features, X_train_embedding_df], axis=1)
    X_val_final = pd.concat([X_val_other_features, X_val_embedding_df], axis=1)

    # Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5,
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # Train the model
    model.fit(X_train_final, y_train_fold)

    # Make predictions
    pred = model.predict(X_val_final)

    # Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

    # Store feature importances for this fold
    all_feature_importances.append(model.feature_importances_)

# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Calculate average feature importances across all folds
mean_feature_importances = np.mean(all_feature_importances, axis=0)

# Create a DataFrame for average feature importances
importance_df_gkf = pd.DataFrame({
    'Feature': full_feature_names,
    'Importance': mean_feature_importances
}).sort_values(by='Importance', ascending=False)

# Print the results
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

# Print top 20 features
N = 20
print(f"\nTop {N} Average Feature Importances (GroupKFold with Embeddings):")
print(importance_df_gkf.head(N))

# Visualize top 20 features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_gkf.head(N), palette='viridis')
plt.title(f'Top {N} Average Feature Importances (GroupKFold LGBMClassifier with Embeddings)')
plt.xlabel('Average Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
#LGBM with sentence embedding data and other 6 vital data
from sentence_transformers import SentenceTransformer

# Load the pre-trained 'all-MiniLM-L6-v2' model
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np


groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# List to store accuracy scores for each fold
accuracy_scores = []


# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X, y, groups=groups)):
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    # Corrected: Initialize X_train_fold and X_val_fold from the original X
    # Use .copy() to avoid SettingWithCopyWarning later
    X_train_fold_raw = X.iloc[train_index].copy()
    X_val_fold_raw = X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    # Process X_train_fold_raw
    X_train_fold_raw['chief_complaint_raw_embedding'] = X_train_fold_raw['chief_complaint_raw'].apply(lambda x: model_sbert.encode(x))
    X_train_fold_only_embeddings = pd.DataFrame(X_train_fold_raw['chief_complaint_raw_embedding'].tolist(), index=X_train_fold_raw.index)
    # Extract other features from the current training fold (not global X)
    X_train_fold_other_features = X_train_fold_raw[['gcs_total','pain_score','news2_score','spo2','respiratory_rate','mean_arterial_pressure']]
    X_train_fold_final = pd.concat([X_train_fold_only_embeddings, X_train_fold_other_features], axis=1)

    # Process X_val_fold_raw
    X_val_fold_raw['chief_complaint_raw_embedding'] = X_val_fold_raw['chief_complaint_raw'].apply(lambda x: model_sbert.encode(x))
    X_val_fold_only_embeddings = pd.DataFrame(X_val_fold_raw['chief_complaint_raw_embedding'].tolist(), index=X_val_fold_raw.index)
    # Extract other features from the current validation fold (not global X)
    X_val_fold_other_features = X_val_fold_raw[['gcs_total','pain_score','news2_score','spo2','respiratory_rate','mean_arterial_pressure']]
    X_val_fold_final = pd.concat([X_val_fold_only_embeddings, X_val_fold_other_features], axis=1)

    # Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5, # Assuming 5 classes for triage_acuity
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # Train the model
    model.fit(X_train_fold_final, y_train_fold)

    # Make predictions
    pred = model.predict(X_val_fold_final)

    # Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

# Calculate mean and standard deviation
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelBinarizer
import numpy as np

# --- Confusion Matrix ---
print("\n--- Confusion Matrix (Last Fold) ---")
cm = confusion_matrix(y_val_fold, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
#Group K fold analysis from sentence embedding only
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# Create a DataFrame X_only_embeddings from the 'chief_complaint_raw_embedding' Series
X_only_embeddings = pd.DataFrame(X['chief_complaint_raw_embedding'].tolist(), index=X.index)
print(f"X_only_embeddings shape: {X_only_embeddings.shape}")

# Define the groups for GroupKFold using the original 'chief_complaint_raw' column
# This ensures that all samples with the same chief complaint are in the same fold
groups = X['chief_complaint_raw']

# Initialize GroupKFold
gkf = GroupKFold(n_splits=5)

# List to store accuracy scores for each fold
accuracy_scores = []

print("\nStarting GroupKFold cross-validation with only sentence embeddings...")

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(gkf.split(X_only_embeddings, y, groups=groups)):
    print(f"\n--- Fold {fold+1}/{gkf.n_splits} ---")

    X_train_fold, X_val_fold = X_only_embeddings.iloc[train_index], X_only_embeddings.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    # Initialize LGBMClassifier
    model = LGBMClassifier(
        objective="multiclass",
        num_class=5, # Assuming 5 classes for triage_acuity
        n_estimators=1000,
        learning_rate=0.03,
        random_state=42 # Ensure reproducibility
    )

    # Train the model using only sentence embeddings
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    pred = model.predict(X_val_fold)

    # Calculate accuracy and append
    accuracy = accuracy_score(y_val_fold, pred)
    accuracy_scores.append(accuracy)
    print(f"Fold {fold+1} Accuracy: {accuracy:.4f}")

# Calculate mean and standard deviation
mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores)

# Print the results
print(f"\nAverage Accuracy across {gkf.n_splits} folds: {mean_accuracy:.4f}")
print(f"Standard Deviation of Accuracy: {std_accuracy:.4f}")

Sentence embedding feature only showed high accuracy rate 0.9820.